## Colab GPU와 프로젝트 파일 확인

## 실험 목적과 비교 기준

기존 YOLO 결과는 50 epoch, Faster R-CNN 결과는 20 epoch 학습이라 학습 횟수가 달랐습니다. 두 모델을 비교하기 위해 같은 v1 데이터 분할(train 185장, validation 47장)에서 각각 20 epoch 학습했습니다.

YOLO는 기존 50 epoch 실험을 덮어쓰지 않도록 별도 실험 이름으로 저장했습니다. 두 모델은 mAP@[0.75:0.95], mAP@0.75와 학습 시간을 비교합니다. 결과는 이 데이터 분할과 설정에서 한 번 측정한 값으로 해석합니다.

In [ ]:
import os
from pathlib import Path

import torch

candidates = [Path.cwd(), Path("/content/YacssoK")]
repo = next(
    (p.resolve() for p in candidates if (p / "configs").is_dir() and (p / "src").is_dir()),
    None,
)

if repo is None:
    raise FileNotFoundError("Colab에서 YacssoK 프로젝트 폴더를 찾지 못했어요.")

os.chdir(repo)

print("프로젝트 위치:", Path.cwd())
print("CUDA 사용 가능:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

files = [
    "data/class_map.json",
    "data/annotations/v1.json",
    "data/splits/v0/train.txt",
    "data/splits/v0/val.txt",
    "data/splits/v1/train.txt",
    "data/splits/v1/val.txt",
    "outputs/1028_dataB_yolov8s_v1_clean/best.pt",
]

for name in files:
    print(f"{'있음' if Path(name).is_file() else '없음'}: {name}")

## Colab에 업로드된 데이터 위치 확인

In [ ]:
import shutil
from pathlib import Path

repo = Path("/content/YacssoK")
names = ["train_images", "train_annotations"]
sources = {name: Path("/content") / name for name in names}
missing = [str(path) for path in sources.values() if not path.is_dir()]

if missing:
    raise FileNotFoundError("아직 못 찾은 폴더:\n" + "\n".join(missing))

for name, source in sources.items():
    destination = repo / "data" / "raw" / name
    shutil.copytree(source, destination, dirs_exist_ok=True)
    count = sum(1 for p in destination.rglob("*") if p.is_file())
    print(f"{name}: {count}개 파일 → {destination}")

### Colab 파일 저장 확인

In [ ]:
%cd /content/YacssoK
!uv run --locked python scripts/prepare_data.py --config configs/base.yaml

## 팀에 저장된 v1 분할 목록 가져오기

In [ ]:
import subprocess
from pathlib import Path

repo = Path("/content/YacssoK")

subprocess.run(["git", "fetch", "origin", "choongeun"], cwd=repo, check=True)

for name in ["data/splits/v1/train.txt", "data/splits/v1/val.txt"]:
    content = subprocess.check_output(
        ["git", "show", f"FETCH_HEAD:{name}"],
        cwd=repo,
    )
    path = repo / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(content)
    print(f"{name}: {len(content.decode('utf-8').splitlines())}장")

In [ ]:
%cd /content/YacssoK
!uv run --locked python scripts/clean_labels.py --config configs/base.yaml --version v1

from pathlib import Path

print("v1 라벨 생성:", Path("data/annotations/v1.json").is_file())
print("분할 유지:", len(Path("data/splits/v1/train.txt").read_text().splitlines()), "train /",
      len(Path("data/splits/v1/val.txt").read_text().splitlines()), "validation")

## 비교에 필요한 파일 점검

In [ ]:
from pathlib import Path

checks = {
    "YOLO 체크포인트": "outputs/1028_dataB_yolov8s_v1_clean/best.pt",
    "YOLO 설정": "outputs/1028_dataB_yolov8s_v1_clean/config.yaml",
    "클래스 목록": "data/class_map.json",
    "v1 라벨": "data/annotations/v1.json",
    "v1 학습 분할": "data/splits/v1/train.txt",
    "v1 검증 분할": "data/splits/v1/val.txt",
    "RCNN 설정": "configs/rcnn_baseline.yaml",
}

for label, name in checks.items():
    path = Path(name)
    print(f"{'있음' if path.is_file() else '없음'}: {label} — {name}")

## RCNN 학습

In [ ]:
import pandas as pd

from src.interfaces import OUTPUT_DIR
from src.train import run
from src.utils import load_config

rcnn_cfg = load_config("configs/rcnn_v1_compare20.yaml")
rcnn_checkpoint = OUTPUT_DIR / rcnn_cfg["exp_name"] / "best.pt"

if rcnn_checkpoint.exists():
    print("RCNN 체크포인트가 이미 있어 학습을 건너뜁니다:", rcnn_checkpoint)
else:
    rcnn_result = run(rcnn_cfg)
    display(pd.DataFrame([rcnn_result]))

## YOLOv8 학습

In [ ]:
# 기존 50 epoch 결과를 보존하고 20 epoch 비교 결과를 별도 저장
%cd /content/YacssoK
!uv run --locked python scripts/run_train.py --config configs/yolo_v1_compare20.yaml 

## 비교 그래프

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ["Faster R-CNN", "YOLOv8s"]
map_075_095 = [0.73839, 0.80109]
map_75 = [0.83801, 0.86084]
train_minutes = [3.38, 0.72]

x = np.arange(len(models))
width = 0.34

fig, (ax_map, ax_time) = plt.subplots(
    1, 2, figsize=(12, 4.5),
    gridspec_kw={"width_ratios": [1.7, 1]}
)

# mAP 비교
bars1 = ax_map.bar(x - width / 2, map_075_095, width, label="mAP@[0.75:0.95]")
bars2 = ax_map.bar(x + width / 2, map_75, width, label="mAP@0.75")

ax_map.set_title("Validation mAP (same v1 split, 20 epochs)")
ax_map.set_ylabel("Score")
ax_map.set_xticks(x, models)
ax_map.set_ylim(0, 1)
ax_map.legend()
ax_map.bar_label(bars1, fmt="%.3f", padding=3)
ax_map.bar_label(bars2, fmt="%.3f", padding=3)

# 학습 시간 비교
bars_time = ax_time.bar(models, train_minutes, color=["#8da0cb", "#66c2a5"])
ax_time.set_title("Training time")
ax_time.set_ylabel("Minutes")
ax_time.bar_label(bars_time, fmt="%.2f min", padding=3)

plt.tight_layout()
plt.show()

| 비교 항목 | Faster R-CNN | YOLOv8s | YOLO 차이 |
|---|---:|---:|---:|
| mAP@[0.75:0.95] | 0.73839 | 0.80109 | +0.06270, 약 **6.27%p** |
| mAP@0.75 | 0.83801 | 0.86084 | +0.02283, 약 **2.28%p** |
| 학습 시간 | 3.38분 | 0.72분 | 약 **4.7배 빠름** |

같은 v1 데이터 분할에서 각 모델을 20 epoch 학습해 비교한 결과, YOLOv8s의 mAP@[0.75:0.95]와 mAP@0.75가 Faster R-CNN보다 각각 0.06270, 0.02283 높았다. 학습 시간은 YOLOv8s가 0.72분, Faster R-CNN이 3.38분으로 측정되어 이번 실험 조건에서는 YOLOv8s가 성능과 학습 시간 모두 유리했다.

### html 파일변환

In [ ]:
import base64
from io import BytesIO
from pathlib import Path

# 위 그래프 셀에서 만든 fig를 HTML 안에 이미지로 넣기
image_buffer = BytesIO()
fig.savefig(image_buffer, format="png", dpi=160, bbox_inches="tight")
chart_base64 = base64.b64encode(image_buffer.getvalue()).decode("utf-8")

html = f"""
<!doctype html>
<html lang="ko">
<head>
  <meta charset="utf-8">
  <title>YOLOv8s와 Faster R-CNN 비교</title>
  <style>
    body {{ font-family: Arial, sans-serif; max-width: 1000px; margin: 40px auto; line-height: 1.6; }}
    img {{ width: 100%; height: auto; }}
  </style>
</head>
<body>
  <h1>YOLOv8s와 Faster R-CNN 비교</h1>
  <p>같은 v1 데이터 분할에서 두 모델을 각각 20 epoch 학습해 비교했습니다.</p>
  <img src="data:image/png;base64,{chart_base64}" alt="모델별 mAP 및 학습 시간 비교 그래프">
  <p>이번 실험에서는 YOLOv8s의 두 mAP 점수가 더 높았고, 학습 시간은 더 짧았습니다.
  검증 데이터 47장으로 진행한 한 번의 비교 결과이므로, 이번 데이터 분할과 설정에서의 결과로 해석합니다.</p>
</body>
</html>
"""

output_path = Path("/content/YacssoK/reports/model_comparison_v1_20epochs.html")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(html, encoding="utf-8")

print("저장 완료:", output_path)